# tinyLMTune — Text Classification

This notebook demonstrates 3 ways to train TinyBERT for **classification** using tinyLMTune:

1. **Synthetic data** — auto-generated via Flan-T5/Mistral
2. **Benchmark data** — real HuggingFace dataset (rotten_tomatoes)
3. **Raw user data** — your own text, structured or unstructured

Each example runs the full pipeline: data → token analysis → search space recommendation → GA optimisation → model save → inference.

## Setup

In [1]:
import logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(name)s | %(message)s")

# Install if needed (uncomment):
!pip install -e ../../tinylmtune_v2/

Obtaining fi
  Preparing metadata (setup.py) ... done
  Using cached datasets-3.6.0-py3-none-any.whl.metadata (19 kB)
  Using cached dill-0.3.8-py3-none-any.whl.metadata (10 kB)
  Using cached multiprocess-0.70.16-py311-none-any.whl.metadata (7.2 kB)
Using cached datasets-3.6.0-py3-none-any.whl (491 kB)
Using cached dill-0.3.8-py3-none-any.whl (116 kB)
Using cached multiprocess-0.70.16-py311-none-any.whl (143 kB)
  Attempting uninstall: dill
    Found existing installation: dill 0.4.0
    Uninstalling dill-0.4.0:
      Successfully uninstalled dill-0.4.0
  Attempting uninstall: multiprocess
    Found existing installation: multiprocess 0.70.18
    Uninstalling multiprocess-0.70.18:
      Successfully uninstalled multiprocess-0.70.18
  Attempting uninstall: datasets
    Found existing installation: datasets 2.2.1
    Uninstalling datasets-2.2.1:
      Successfully uninstalled datasets-2.2.1
  DEPRECATION: Legacy editable install of tinylmtune==2.0.0 from file:///home/sagemaker-user/SLM/

In [1]:
from tinylmtune import optimize_slm, TinyInference, print_token_analysis, print_recommendation, plot_results

2026-05-25 07:09:31.099722: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-05-25 07:09:31.112125: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-05-25 07:09:31.115976: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1452] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-05-25 07:09:31.125608: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE4.1 SSE4.2 AVX AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
## Check search space recommendation before you start training 

print_recommendation(n_samples=200, task="classification")

  tinyLMTune — Search Space Recommendation
  Task: classification  |  Samples: 200  |  Max len: 128
  Parameters: 11

  Recommended search_space:
    learning_rate                   : (1e-05, 0.0005)
    batch_size                      : [4, 8, 16, 32]
    epochs                          : (3, 15)
    warmup_ratio                    : (0.0, 0.3)
    weight_decay                    : (0.0, 0.1)
    dropout                         : (0.05, 0.3)
    attention_dropout               : (0.05, 0.3)
    gradient_accumulation_steps     : [1, 2, 4, 8]
    lr_scheduler_type               : ['linear', 'cosine', 'cosine_with_restarts', 'constant_with_warmup']
    label_smoothing                 : (0.0, 0.1)
    max_grad_norm                   : (0.5, 5.0)



{'learning_rate': (1e-05, 0.0005),
 'batch_size': [4, 8, 16, 32],
 'epochs': (3, 15),
 'warmup_ratio': (0.0, 0.3),
 'weight_decay': (0.0, 0.1),
 'dropout': (0.05, 0.3),
 'attention_dropout': (0.05, 0.3),
 'gradient_accumulation_steps': [1, 2, 4, 8],
 'lr_scheduler_type': ['linear',
  'cosine',
  'cosine_with_restarts',
  'constant_with_warmup'],
 'label_smoothing': (0.0, 0.1),
 'max_grad_norm': (0.5, 5.0)}

---
## Example 1 — Synthetic Data (via Flan-T5)

No data needed. Flan-T5/Mistral generates training data from a topic prompt.

**Requirements:** Flan-T5 must be installed and running (`pip install sentencepiece`), with `mistral` model pulled (``).

In [3]:
best = optimize_slm(
    task="classification",
    corpus_prompt="Generate diverse product review sentiment examples",
    n_examples=200,
    labels="positive,negative,neutral",
    pop_size=4,
    generations=5,
    output_dir="models/classification_synthetic",
)
#print("Best config:", best)



Some weights of BertForSequenceClassification were not initialized from the model checkpoint at huawei-noah/TinyBERT_General_4L_312D and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,1.097300,1.097315,0.350000,0.265116
2,1.095200,1.092706,0.400000,0.279210
3,1.086800,1.078702,0.400000,0.291515
4,1.068600,1.050157,0.600000,0.527273
5,1.051200,1.025712,0.625000,0.574570
6,1.030300,1.010891,0.600000,0.527273
7,1.021600,1.000669,0.675000,0.609378
8,1.013700,0.999871,0.675000,0.609378


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at huawei-noah/TinyBERT_General_4L_312D and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.916000,1.111490,0.300000,0.138462
2,0.910500,1.028649,0.550000,0.499802
3,0.873100,0.980509,0.600000,0.553853
4,0.835100,1.001878,0.600000,0.540209
5,0.760300,0.897797,0.625000,0.538143


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at huawei-noah/TinyBERT_General_4L_312D and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


...

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,1.086700,1.077673,0.425000,0.344525
2,1.090500,1.027429,0.525000,0.491806
3,0.994500,0.937954,0.650000,0.579653
4,0.943500,0.926775,0.600000,0.521622
5,0.879100,1.002588,0.550000,0.486305


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at huawei-noah/TinyBERT_General_4L_312D and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,1.076100,1.051540,0.500000,0.457365
2,1.011400,0.929971,0.650000,0.589286
3,0.933400,0.908278,0.675000,0.629333
4,0.830200,0.899683,0.575000,0.519015
5,0.801700,0.926060,0.600000,0.561520


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at huawei-noah/TinyBERT_General_4L_312D and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,1.076100,1.051540,0.500000,0.457365
2,1.011400,0.929971,0.650000,0.589286
3,0.933400,0.908278,0.675000,0.629333
4,0.830200,0.899683,0.575000,0.519015
5,0.801700,0.926060,0.600000,0.561520


In [4]:
# Only displays, doesn't save
plot_results(best, save_dir="plots/synthetic_data")

{'fitness_progress': <Figure size 1000x600 with 1 Axes>,
 'parameter_scatter': <Figure size 1800x1800 with 12 Axes>,
 'scheduler_comparison': <Figure size 800x500 with 1 Axes>,
 'config_evolution': <Figure size 1800x1800 with 12 Axes>,
 'population_heatmap': <Figure size 1000x400 with 2 Axes>}

### Inference on synthetic model

In [5]:
model = TinyInference("models/classification_synthetic")

for text in [
    "Absolutely love this product, best purchase ever!",
    "Terrible quality, broke after one day.",
    "It's fine, does what it says.",
]:
    result = model.predict(text)
    print(f"Text:  {text}")
    print(f"Label: {result['label']}  Confidence: {result['confidence']:.1%}")
    print()


Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


Text:  Absolutely love this product, best purchase ever!
Label: positive  Confidence: 67.5%

Text:  Terrible quality, broke after one day.
Label: negative  Confidence: 60.5%

Text:  It's fine, does what it says.
Label: negative  Confidence: 47.2%



---
## Example 2 — Benchmark Data (rotten_tomatoes)

Uses a real HuggingFace dataset. No Flan-T5 needed.

### Load rotten_tomatoes dataset

In [6]:
from datasets import load_dataset

ds = load_dataset("rotten_tomatoes", split="train")
ds = ds.shuffle(seed=42).select(range(1000))  # use 300 samples

label_map = {0: "negative", 1: "positive"}
benchmark_data = [{"text": r["text"], "label": label_map[r["label"]]} for r in ds]

print(f"Loaded {len(benchmark_data)} records")
print(f"Sample: {benchmark_data[0]}")

Loaded 1000 records
Sample: {'text': '. . . plays like somebody spliced random moments of a chris rock routine into what is otherwise a cliche-riddled but self-serious spy thriller .', 'label': 'negative'}


### Analyze token lengths

In [7]:
print_token_analysis(benchmark_data, task="classification")

  tinyLMTune — Token Length Analysis
  Task: classification  |  Samples: 1000

  Token lengths:  min=3  mean=27  median=27  max=78

  Percentiles:
    p 50:   27 tokens
    p 75:   34 tokens
    p 90:   43 tokens
    p 95:   48 tokens
    p100:   78 tokens

  Truncation at standard lengths:
    max_len=  2: 100.0% truncated  ██████████████████████████████████████████████████
    max_len=  4:  99.9% truncated  █████████████████████████████████████████████████
    max_len=  8:  97.5% truncated  ████████████████████████████████████████████████
    max_len= 16:  83.2% truncated  █████████████████████████████████████████
    max_len= 24:  55.7% truncated  ███████████████████████████
    max_len= 32:  30.5% truncated  ███████████████
    max_len= 64:   0.2% truncated   ← recommended
    max_len= 96:   0.0% truncated  
    max_len=128:   0.0% truncated  
    max_len=192:   0.0% truncated  
    max_len=256:   0.0% truncated  
    max_len=384:   0.0% truncated  
    max_len=512:   0.0% truncate

{'n_samples': 1000,
 'min': 3,
 'max': 78,
 'mean': 27,
 'median': 27,
 'p50': 27,
 'p75': 34,
 'p90': 43,
 'p95': 48,
 'p100': 78,
 'truncation_pct': {2: 100.0,
  4: 99.9,
  8: 97.5,
  16: 83.2,
  24: 55.7,
  32: 30.5,
  64: 0.2,
  96: 0.0,
  128: 0.0,
  192: 0.0,
  256: 0.0,
  384: 0.0,
  512: 0.0},
 'recommended_max_len': 64,
 'recommended_ga_choices': [32, 64, 96]}

### Check recommended search space

In [8]:
print_recommendation(n_samples=len(benchmark_data), task="classification")

  tinyLMTune — Search Space Recommendation
  Task: classification  |  Samples: 1000  |  Max len: 128
  Parameters: 11

  Recommended search_space:
    learning_rate                   : (5e-06, 0.0005)
    batch_size                      : [4, 8, 16, 32]
    epochs                          : (2, 10)
    warmup_ratio                    : (0.0, 0.3)
    weight_decay                    : (0.0, 0.05)
    dropout                         : (0.0, 0.2)
    attention_dropout               : (0.0, 0.2)
    gradient_accumulation_steps     : [1, 2, 4, 8]
    lr_scheduler_type               : ['linear', 'cosine', 'cosine_with_restarts', 'constant_with_warmup']
    label_smoothing                 : (0.0, 0.1)
    max_grad_norm                   : (0.5, 5.0)



{'learning_rate': (5e-06, 0.0005),
 'batch_size': [4, 8, 16, 32],
 'epochs': (2, 10),
 'warmup_ratio': (0.0, 0.3),
 'weight_decay': (0.0, 0.05),
 'dropout': (0.0, 0.2),
 'attention_dropout': (0.0, 0.2),
 'gradient_accumulation_steps': [1, 2, 4, 8],
 'lr_scheduler_type': ['linear',
  'cosine',
  'cosine_with_restarts',
  'constant_with_warmup'],
 'label_smoothing': (0.0, 0.1),
 'max_grad_norm': (0.5, 5.0)}

### Train with GA optimisation

In [25]:
best = optimize_slm(
    task="classification",
    user_data=benchmark_data,
    pop_size=6,
    generations=3,
    max_len = 64,
    output_dir="models/classification_benchmark",
)
print("Best config:", best)

2026-05-15 11:27:23,269 | tinylmtune._internal.pipeline | Model will be saved to: /home/sagemaker-user/SLM/tinylmtune_v2/notebooks/models/classification_benchmark
2026-05-15 11:27:23,270 | tinylmtune._internal.pipeline | user_data validated: 1000 / 1000 structured records
2026-05-15 11:27:23,270 | tinylmtune._internal.pipeline | Resolved 1000 structured records
2026-05-15 11:27:23,593 | tinylmtune._internal.token_analyzer | Token analysis: n=1000 min=3 mean=27 p95=48 max=78 → max_len=64, ga_choices=[32, 64, 96]
2026-05-15 11:27:23,598 | tinylmtune._internal.pipeline | Token analysis: p50=27 p95=48 max=78 → fixed max_len=64
2026-05-15 11:27:23,599 | tinylmtune._internal.dataset | Using 1000 user-provided records
2026-05-15 11:27:23,869 | tinylmtune._internal.dataset | Dataset: 800 train, 200 val
2026-05-15 11:27:23,900 | tinylmtune._internal.pipeline | Fixed max_len=64 | GA search space (11 params): ['learning_rate', 'batch_size', 'epochs', 'warmup_ratio', 'weight_decay', 'dropout', 'at

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.682200,0.699422,0.475000,0.305932
2,0.694200,0.694382,0.475000,0.305932
3,0.693900,0.694003,0.475000,0.305932


2026-05-15 11:27:46,529 | tinylmtune._internal.trainer | Eval metrics: {'eval_loss': 0.6994220614433289, 'eval_accuracy': 0.475, 'eval_f1': 0.30593220338983046, 'eval_runtime': 0.6465, 'eval_samples_per_second': 309.34, 'eval_steps_per_second': 77.335, 'epoch': 3.0}
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at huawei-noah/TinyBERT_General_4L_312D and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
2026-05-15 11:27:46,730 | tinylmtune._internal.trainer | Training: lr=0.0001201671422284161 bs=4 epochs=10 dropout=0.11 attn_drop=0.04 grad_accum=4 scheduler=linear label_smooth=0.076 grad_norm=1.2


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.648200,0.513432,0.795000,0.794799
2,0.522400,0.546483,0.765000,0.761081
3,0.417800,0.509668,0.780000,0.778543


2026-05-15 11:28:05,344 | tinylmtune._internal.trainer | Eval metrics: {'eval_loss': 0.5134322643280029, 'eval_accuracy': 0.795, 'eval_f1': 0.794799276945093, 'eval_runtime': 0.657, 'eval_samples_per_second': 304.402, 'eval_steps_per_second': 76.1, 'epoch': 3.0}
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at huawei-noah/TinyBERT_General_4L_312D and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
2026-05-15 11:28:05,572 | tinylmtune._internal.trainer | Training: lr=0.00021419412708598376 bs=16 epochs=4 dropout=0.02 attn_drop=0.08 grad_accum=4 scheduler=cosine_with_restarts label_smooth=0.060 grad_norm=4.1


...

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.657300,0.541335,0.760000,0.756364
2,0.587200,0.508439,0.790000,0.789243
3,0.449900,0.504054,0.805000,0.805044
4,0.345100,0.485603,0.810000,0.809886
5,0.272000,0.633137,0.785000,0.785048
6,0.232500,0.634350,0.805000,0.805122


2026-05-15 11:35:42,001 | tinylmtune._internal.trainer | Eval metrics: {'eval_loss': 0.48560279607772827, 'eval_accuracy': 0.81, 'eval_f1': 0.809885588117222, 'eval_runtime': 0.3648, 'eval_samples_per_second': 548.174, 'eval_steps_per_second': 68.522, 'epoch': 6.0}
2026-05-15 11:35:42,002 | tinylmtune._internal.ga_optimizer | Gen 3/3 — best=0.8197  avg=0.7797  worst=0.7147
2026-05-15 11:35:42,003 | tinylmtune._internal.pipeline | Best config (fitness=0.8197): {'learning_rate': 0.0001201671422284161, 'batch_size': 4, 'epochs': 10, 'warmup_ratio': 0.06, 'weight_decay': 0.065, 'dropout': 0.109, 'attention_dropout': 0.044, 'gradient_accumulation_steps': 4, 'lr_scheduler_type': 'linear', 'label_smoothing': 0.076, 'max_grad_norm': 1.22, 'fitness': 0.8197467591196864}
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at huawei-noah/TinyBERT_General_4L_312D and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN t

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.645700,0.525210,0.765000,0.765053
2,0.521700,0.515305,0.770000,0.770138
3,0.455400,0.612134,0.760000,0.756317
4,0.384200,0.596009,0.780000,0.779690
5,0.285000,0.654990,0.760000,0.757461
6,0.248500,0.768678,0.740000,0.736061


2026-05-15 11:36:12,124 | tinylmtune._internal.trainer | Eval metrics: {'eval_loss': 0.5960086584091187, 'eval_accuracy': 0.78, 'eval_f1': 0.7796904833685057, 'eval_runtime': 0.6764, 'eval_samples_per_second': 295.682, 'eval_steps_per_second': 73.92, 'epoch': 6.0}
2026-05-15 11:36:12,277 | tinylmtune._internal.inference | Model saved → /home/sagemaker-user/SLM/tinylmtune_v2/notebooks/models/classification_benchmark


Best config: {'learning_rate': 0.0001201671422284161, 'batch_size': 4, 'epochs': 10, 'warmup_ratio': 0.06, 'weight_decay': 0.065, 'dropout': 0.109, 'attention_dropout': 0.044, 'gradient_accumulation_steps': 4, 'lr_scheduler_type': 'linear', 'label_smoothing': 0.076, 'max_grad_norm': 1.22, 'fitness': 0.8197467591196864, 'max_len': 64, 'task': 'classification', 'output_dir': '/home/sagemaker-user/SLM/tinylmtune_v2/notebooks/models/classification_benchmark', 'ga_history': [{'generation': 1, 'individual': 0, 'fitness': 0.30593220338983046, 'learning_rate': 0.00032151626523665245, 'batch_size': 4, 'epochs': 6, 'warmup_ratio': 0.073, 'weight_decay': 0.014, 'dropout': 0.02, 'attention_dropout': 0.148, 'gradient_accumulation_steps': 1, 'lr_scheduler_type': 'constant_with_warmup', 'label_smoothing': 0.003, 'max_grad_norm': 0.92}, {'generation': 1, 'individual': 1, 'fitness': 0.794799276945093, 'learning_rate': 0.0001201671422284161, 'batch_size': 4, 'epochs': 10, 'warmup_ratio': 0.06, 'weight_d

### Visualize GA Results

5 plots showing how the GA searched for the best hyperparameters:
1. **Fitness progress** — best/avg/worst per generation
2. **Parameter scatter** — each param vs fitness (best = red star)
3. **Scheduler comparison** — box plot by LR scheduler type
4. **Config evolution** — how the best config changed over generations
5. **Population heatmap** — all individuals in the last generation

In [26]:
from tinylmtune import plot_results, print_best_config_table

# Print formatted best config
print_best_config_table(best)

# Generate all 5 plots
figs = plot_results(best, save_dir="plots/benchmark")

2026-05-15 11:36:47,023 | tinylmtune._internal.visualizer | Saved: plots/benchmark/01_fitness_progress.png


  Best Configuration — F1 Score: 0.8197
  learning_rate                      : 0.000120167
  batch_size                         : 4
  epochs                             : 10
  warmup_ratio                       : 0.06
  weight_decay                       : 0.065
  dropout                            : 0.109
  attention_dropout                  : 0.044
  gradient_accumulation_steps        : 4
  lr_scheduler_type                  : linear
  label_smoothing                    : 0.076
  max_grad_norm                      : 1.22
  max_len (fixed from data)          : 64
  output_dir                         : /home/sagemaker-user/SLM/tinylmtune_v2/notebooks/models/classification_benchmark



2026-05-15 11:36:48,417 | tinylmtune._internal.visualizer | Saved: plots/benchmark/02_parameter_scatter.png
2026-05-15 11:36:48,541 | tinylmtune._internal.visualizer | Saved: plots/benchmark/03_scheduler_comparison.png
2026-05-15 11:36:49,865 | tinylmtune._internal.visualizer | Saved: plots/benchmark/04_config_evolution.png
2026-05-15 11:36:50,122 | tinylmtune._internal.visualizer | Saved: plots/benchmark/05_population_heatmap.png
2026-05-15 11:36:50,123 | tinylmtune._internal.visualizer | Generated 5 plots


### Inference on benchmark model

In [28]:
model = TinyInference("models/classification_benchmark")

test_texts = [
    "A stunning visual masterpiece with brilliant performances.",
    "Predictable plot and terrible dialogue throughout.",
    "An average film that neither excels nor disappoints.",
]
for text in test_texts:
    result = model.predict(text)
    print(f"{result['label']:10s} ({result['confidence']:.1%})  {text}")

2026-05-15 11:37:22,995 | tinylmtune._internal.inference | Loaded classification model from models/classification_benchmark
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


positive   (94.3%)  A stunning visual masterpiece with brilliant performances.
positive   (60.6%)  Predictable plot and terrible dialogue throughout.
negative   (93.7%)  An average film that neither excels nor disappoints.


---
## Example 3 — Raw User Data

Three sub-examples showing different input formats:
- **3a.** Structured dicts (correct format)
- **3b.** Raw text strings (auto-labelled via Flan-T5)
- **3c.** Wrong-format dicts (auto-detected and converted)

### 3a. Structured dicts (used directly, no Flan-T5)

In [15]:
my_data = [
    {"text": "Absolutely loved this product!", "label": "positive"},
    {"text": "Best purchase I've made this year", "label": "positive"},
    {"text": "Works great, highly recommend", "label": "positive"},
    {"text": "Exceeded all my expectations", "label": "positive"},
    {"text": "Amazing quality for the price", "label": "positive"},
    {"text": "Will definitely buy again", "label": "positive"},
    {"text": "Perfect gift, they loved it", "label": "positive"},
    {"text": "Five stars, no complaints at all", "label": "positive"},
    {"text": "Total waste of money", "label": "negative"},
    {"text": "Broke after just two days", "label": "negative"},
    {"text": "Worst product I've ever bought", "label": "negative"},
    {"text": "Completely unusable, returning it", "label": "negative"},
    {"text": "Poor quality, very disappointed", "label": "negative"},
    {"text": "Do not buy, save your money", "label": "negative"},
    {"text": "Terrible customer service too", "label": "negative"},
    {"text": "Fell apart on first use", "label": "negative"},
    {"text": "It's okay, nothing special", "label": "neutral"},
    {"text": "Average product, does the job", "label": "neutral"},
    {"text": "Meets expectations, no more no less", "label": "neutral"},
    {"text": "Fine for the price, not amazing", "label": "neutral"},
]

best = optimize_slm(
    task="classification",
    user_data=my_data,
    pop_size=4,
    generations=2,
    output_dir="models/classification_user",
)

2026-05-15 08:41:55,267 | tinylmtune._internal.pipeline | Model will be saved to: /home/sagemaker-user/SLM/tinylmtune_v2/notebooks/models/classification_user
2026-05-15 08:41:55,268 | tinylmtune._internal.pipeline | user_data validated: 20 / 20 structured records
2026-05-15 08:41:55,268 | tinylmtune._internal.pipeline | Resolved 20 structured records
2026-05-15 08:41:55,478 | tinylmtune._internal.token_analyzer | Token analysis: n=20 min=6 mean=7 p95=9 max=10 → max_len=32, ga_choices=[32]
2026-05-15 08:41:55,481 | tinylmtune._internal.pipeline | Token analysis: p50=7 p95=9 max=10 → fixed max_len=32
2026-05-15 08:41:55,482 | tinylmtune._internal.dataset | Using 20 user-provided records
2026-05-15 08:41:55,668 | tinylmtune._internal.dataset | Dataset: 16 train, 4 val
2026-05-15 08:41:55,670 | tinylmtune._internal.pipeline | Fixed max_len=32 | GA search space (11 params): ['learning_rate', 'batch_size', 'epochs', 'warmup_ratio', 'weight_decay', 'dropout', 'attention_dropout', 'gradient_ac

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,1.097200,1.106756,0.000000,0.000000
2,1.098300,1.101293,0.250000,0.100000
3,1.090500,1.096411,0.250000,0.100000
4,1.080600,1.093712,0.250000,0.100000


2026-05-15 08:41:57,552 | tinylmtune._internal.trainer | Eval metrics: {'eval_loss': 1.1012930870056152, 'eval_accuracy': 0.25, 'eval_f1': 0.1, 'eval_runtime': 0.0083, 'eval_samples_per_second': 483.535, 'eval_steps_per_second': 120.884, 'epoch': 4.0}
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at huawei-noah/TinyBERT_General_4L_312D and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
2026-05-15 08:41:57,721 | tinylmtune._internal.trainer | Training: lr=2.7802095573702566e-05 bs=4 epochs=5 dropout=0.31 attn_drop=0.23 grad_accum=2 scheduler=cosine_with_restarts label_smooth=0.162 grad_norm=0.5


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,1.097200,1.095111,0.750000,0.642857
2,1.096700,1.095360,0.500000,0.500000
3,1.096500,1.095156,0.250000,0.300000


2026-05-15 08:42:01,316 | tinylmtune._internal.trainer | Eval metrics: {'eval_loss': 1.0951111316680908, 'eval_accuracy': 0.75, 'eval_f1': 0.6428571428571428, 'eval_runtime': 0.0077, 'eval_samples_per_second': 517.464, 'eval_steps_per_second': 129.366, 'epoch': 3.0}
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at huawei-noah/TinyBERT_General_4L_312D and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
2026-05-15 08:42:01,479 | tinylmtune._internal.trainer | Training: lr=0.00016310565784823352 bs=8 epochs=15 dropout=0.33 attn_drop=0.13 grad_accum=2 scheduler=linear label_smooth=0.072 grad_norm=2.0


...

2026-05-15 08:42:23,934 | tinylmtune._internal.trainer | Eval metrics: {'eval_loss': 1.0988287925720215, 'eval_accuracy': 0.25, 'eval_f1': 0.1, 'eval_runtime': 0.0076, 'eval_samples_per_second': 525.981, 'eval_steps_per_second': 131.495, 'epoch': 5.0}
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at huawei-noah/TinyBERT_General_4L_312D and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
2026-05-15 08:42:24,104 | tinylmtune._internal.trainer | Training: lr=7.750143304322909e-05 bs=4 epochs=19 dropout=0.21 attn_drop=0.24 grad_accum=4 scheduler=cosine_with_restarts label_smooth=0.115 grad_norm=2.0


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,1.097700,1.106784,0.000000,0.000000
2,1.098500,1.105200,0.000000,0.000000
3,1.097600,1.102463,0.250000,0.100000
4,1.094800,1.098506,0.250000,0.100000
5,1.089800,1.095421,0.250000,0.100000


2026-05-15 08:42:30,346 | tinylmtune._internal.trainer | Eval metrics: {'eval_loss': 1.1024627685546875, 'eval_accuracy': 0.25, 'eval_f1': 0.1, 'eval_runtime': 0.0097, 'eval_samples_per_second': 411.681, 'eval_steps_per_second': 102.92, 'epoch': 5.0}
2026-05-15 08:42:30,348 | tinylmtune._internal.ga_optimizer | Gen 2/2 — best=0.1000  avg=0.0500  worst=0.0000
2026-05-15 08:42:30,348 | tinylmtune._internal.pipeline | Best config (fitness=0.6429): {'learning_rate': 2.7802095573702566e-05, 'batch_size': 4, 'epochs': 5, 'warmup_ratio': 0.112, 'weight_decay': 0.1074, 'dropout': 0.31, 'attention_dropout': 0.226, 'gradient_accumulation_steps': 2, 'lr_scheduler_type': 'cosine_with_restarts', 'label_smoothing': 0.162, 'max_grad_norm': 0.53, 'fitness': 0.6428571428571428}
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at huawei-noah/TinyBERT_General_4L_312D and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN t

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,1.098900,1.106104,0.000000,0.000000
2,1.100700,1.105758,0.000000,0.000000
3,1.096400,1.104996,0.250000,0.100000
4,1.093800,1.104345,0.250000,0.100000
5,1.096900,1.104246,0.250000,0.100000


2026-05-15 08:42:32,205 | tinylmtune._internal.trainer | Eval metrics: {'eval_loss': 1.1049959659576416, 'eval_accuracy': 0.25, 'eval_f1': 0.1, 'eval_runtime': 0.0081, 'eval_samples_per_second': 496.794, 'eval_steps_per_second': 124.198, 'epoch': 5.0}
2026-05-15 08:42:32,285 | tinylmtune._internal.inference | Model saved → /home/sagemaker-user/SLM/tinylmtune_v2/notebooks/models/classification_user


### 3b. Raw text strings (requires Flan-T5)

In [ ]:
# Raw strings — Flan-T5 will label them automatically
raw_texts = [
    "This movie was absolutely wonderful and heartwarming!",
    "A complete waste of two hours of my life.",
    "The acting was superb but the plot dragged on.",
    "One of the best films I've seen this year!",
    "Terribly written with no character development.",
    "Decent entertainment, nothing groundbreaking.",
]

labels = "positive,negative,neutral"

best = optimize_slm(
    task="classification",
    user_data=raw_texts,
    labels= labels,
    pop_size=4,
    generations=1,
    output_dir="models/classification_raw",
)

### 3c. Wrong-format dicts (requires Flan-T5)

In [ ]:
# Dicts with wrong keys — pipeline auto-detects and converts
wrong_format = [
    {"content": "This restaurant has amazing food and great service!"},
    {"content": "Worst dining experience ever, never going back."},
    {"content": "The food was okay but overpriced for the quality."},
    {"content": "Absolutely delicious, my new favourite place!"},
    {"content": "Rude staff and cold food, very disappointed."},
    {"content": "Average meal, nothing to write home about."},
]

# Pipeline detects "content" key, extracts text, labels via Flan-T5
best = optimize_slm(
    task="classification",
    user_data=wrong_format,
    labels="positive,negative,neutral",
    pop_size=4,
    generations=1,
    output_dir="models/classification_wrong",
)

### Visualize user data results

In [16]:
# Plot results from structured data training (Example 3a)
from tinylmtune import plot_results, print_best_config_table
print_best_config_table(best)
figs = plot_results(best, save_dir="plots/user_data")

2026-05-15 08:44:27,668 | tinylmtune._internal.visualizer | Saved: plots/user_data/01_fitness_progress.png


  Best Configuration — F1 Score: 0.6429
  learning_rate                      : 2.78021e-05
  batch_size                         : 4
  epochs                             : 5
  warmup_ratio                       : 0.112
  weight_decay                       : 0.1074
  dropout                            : 0.31
  attention_dropout                  : 0.226
  gradient_accumulation_steps        : 2
  lr_scheduler_type                  : cosine_with_restarts
  label_smoothing                    : 0.162
  max_grad_norm                      : 0.53
  max_len (fixed from data)          : 32
  output_dir                         : /home/sagemaker-user/SLM/tinylmtune_v2/notebooks/models/classification_user



2026-05-15 08:44:28,929 | tinylmtune._internal.visualizer | Saved: plots/user_data/02_parameter_scatter.png
2026-05-15 08:44:29,035 | tinylmtune._internal.visualizer | Saved: plots/user_data/03_scheduler_comparison.png
2026-05-15 08:44:30,348 | tinylmtune._internal.visualizer | Saved: plots/user_data/04_config_evolution.png
2026-05-15 08:44:30,558 | tinylmtune._internal.visualizer | Saved: plots/user_data/05_population_heatmap.png
2026-05-15 08:44:30,558 | tinylmtune._internal.visualizer | Generated 5 plots


### Inference

In [18]:
model = TinyInference("models/classification_user")
result = model.predict("This is the best thing I've ever purchased!")
print(result)

2026-05-15 08:46:37,173 | tinylmtune._internal.inference | Loaded classification model from models/classification_user
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


{'label': 'positive', 'confidence': 0.3405956029891968, 'probabilities': {'negative': 0.32547882199287415, 'neutral': 0.3339255750179291, 'positive': 0.3405956029891968}}


---
## Summary

| Example | Data source | Flan-T5 needed | Best for |
|---------|-------------|---------------|----------|
| Synthetic | Auto-generated | Yes | Quick prototyping |
| Benchmark | rotten_tomatoes | No | Reproducible evaluation |
| User data | Your own text | Depends on format | Production use |

The GA searches 11 hyperparameters: `learning_rate`, `batch_size`, `epochs`, `warmup_ratio`, `weight_decay`, `dropout`, `attention_dropout`, `gradient_accumulation_steps`, `lr_scheduler_type`, `label_smoothing`, `max_grad_norm`.

`max_len` is automatically determined from your data's token length distribution (p95 percentile).